# Notebook 01: Exploratory Data Analysis (EDA) of Type 2 Diabetes Longitudinal Cohort
This notebook explores the healthcare records, biomarker distributions, encounter intervals, and patient demographics for the longitudinal Type 2 Diabetes cohort.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import config
from preprocessing.load_data import load_raw_data

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Longitudinal Cohort Records

In [ ]:
# Load cached interim cohort or generate calibrated cohort
df = load_raw_data(source="auto", num_patients=500)
print(f"Total observations: {len(df)}")
print(f"Unique patients: {df['subject_id'].nunique()}")
df.head(10)

## 2. Demographic and Baseline Patient Profiles

In [ ]:
patient_first = df.groupby("subject_id").first().reset_index()

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(patient_first["age"], bins=15, kde=True, ax=axs[0], color="#1a73e8")
axs[0].set_title("Age Distribution at Baseline")

sns.countplot(x="gender", data=patient_first, ax=axs[1], palette="Blues_r")
axs[1].set_title("Gender Distribution")

sns.countplot(x="smoking_status", data=patient_first, ax=axs[2], palette="Greens_r")
axs[2].set_title("Smoking Status Distribution")
plt.tight_layout()
plt.show()

## 3. Clinical Biomarker Distributions & Guideline Targets

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 8))
biomarkers_to_plot = ["hba1c", "glucose", "bmi", "sbp", "ldl", "creatinine"]

for idx, marker in enumerate(biomarkers_to_plot):
    ax = axs[idx // 3, idx % 3]
    sns.histplot(df[marker].dropna(), kde=True, ax=ax, color="#0d6efd")
    thresh = config.CLINICAL_THRESHOLDS.get(marker, {})
    target = thresh.get("target_max")
    if target:
        ax.axvline(target, color="red", linestyle="--", label=f"Target ({target})")
        ax.legend()
    ax.set_title(f"{marker.upper()} Distribution")

plt.tight_layout()
plt.show()

## 4. Longitudinal Visit Intervals and Follow-Up Patterns

In [ ]:
visit_counts = df.groupby("subject_id")["chartdate"].count()
print("Visits per patient summary:")
print(visit_counts.describe())

plt.figure(figsize=(8, 4))
sns.countplot(x=visit_counts, palette="viridis")
plt.title("Distribution of Total Visits per Patient")
plt.xlabel("Number of Encounters")
plt.ylabel("Number of Patients")
plt.show()